# pretraining

this notebook explores how to pre-train an LLM

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import tiktoken
import torch
import torch.nn.functional as F

from functions import (
    gpt_model,
    create_dataloader_v1,
    generate_text_simple,
)

In [2]:
# creates random weights for the GPT model
# sets the random seed for reproducibility

gpt_config = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

torch.manual_seed(123)

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

tokenizer = tiktoken.get_encoding("gpt2")

model = gpt_model(gpt_config)
model = model.to(device)

print("device:", device)

device: mps


In [5]:
# load the text data

data_path = Path("./data/the-verdict.txt")

text_data = data_path.read_text(
    encoding="utf-8"
)

print("number of characters:", len(text_data))
print(text_data[:200])

number of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a


In [6]:
# creates training and validation datasets

training_ratio = 0.90

split_index = int(
    training_ratio * len(text_data)
)

training_text = text_data[:split_index]
validation_text = text_data[split_index:]

print(
    "training characters:",
    len(training_text),
)

print(
    "validation characters:",
    len(validation_text),
)

training characters: 18431
validation characters: 2048


In [7]:
# create the dataloaders for training and validation

batch_size = 2

training_loader = create_dataloader_v1(
    training_text,
    batch_size=batch_size,
    max_length=gpt_config["context_length"],
    stride=gpt_config["context_length"],
    shuffle=True,
    drop_last=True,
    num_workers=0,
)

validation_loader = create_dataloader_v1(
    validation_text,
    batch_size=batch_size,
    max_length=gpt_config["context_length"],
    stride=gpt_config["context_length"],
    shuffle=False,
    drop_last=False,
    num_workers=0,
)

In [8]:
# inspect one batch of input and target tensors

input_batch, target_batch = next(
    iter(training_loader)
)

print("input shape:", input_batch.shape)
print("target shape:", target_batch.shape)

print("\nfirst five input IDs:")
print(input_batch[0, :5])

print("\nfirst five target IDs:")
print(target_batch[0, :5])

input shape: torch.Size([2, 256])
target shape: torch.Size([2, 256])

first five input IDs:
tensor([  503,  4291,   262,  4252, 18250])

first five target IDs:
tensor([ 4291,   262,  4252, 18250,  8812])


In [9]:
# define the batch-loss function

def calculate_batch_loss(
    input_batch,
    target_batch,
    model,
    device,
):

    # move batches to same device as the model
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)

    # run the forward pass of the model to get the logits
    logits = model(input_batch)

    # calculate the cross-entropy loss between the logits and the target batch
    loss = F.cross_entropy(
        logits.flatten(0, 1),
        target_batch.flatten(),
    )

    return loss

In [10]:
# define the loader-loss function

def calculate_loader_loss(
    data_loader,
    model,
    device,
    number_of_batches=None,
):

    # if the data loader is empty, return NaN
    if len(data_loader) == 0:
        return float("nan")

    # if the number of batches is not specified, use the entire data loader
    if number_of_batches is None:
        number_of_batches = len(data_loader)
    else:
        number_of_batches = min(
            number_of_batches,
            len(data_loader),
        )

    # initialize the total loss to zero
    total_loss = 0.0

    # iterate over the data loader and calculate the loss for each batch
    for batch_number, (
        input_batch,
        target_batch,
    ) in enumerate(data_loader):
        if batch_number >= number_of_batches:
            break

        loss = calculate_batch_loss(
            input_batch,
            target_batch,
            model,
            device,
        )
        # accumulate the loss for each batch
        total_loss += loss.item()

    # return the average loss over the specified number of batches
    return total_loss / number_of_batches

In [11]:
# evaluate the model on the training and validation datasets

model.eval()

# calculate the initial training and validation losses 
# without computing gradients
with torch.no_grad():

    # calculate the initial training loss
    initial_training_loss = (
        calculate_loader_loss(
            training_loader,
            model,
            device,
        )
    )

    # calculate the initial validation loss
    initial_validation_loss = (
        calculate_loader_loss(
            validation_loader,
            model,
            device,
        )
    )

initial_perplexity = torch.exp(
    torch.tensor(initial_validation_loss)
).item()

print(
    "initial training loss:",
    initial_training_loss,
)

print(
    "initial validation loss:",
    initial_validation_loss,
)

print(
    "initial validation perplexity:",
    initial_perplexity,
)

initial training loss: 10.98758316040039
initial validation loss: 10.98110580444336
initial validation perplexity: 58753.48828125


In [13]:
# define the text-to-token-IDs and token-IDs-to-text functions

def text_to_token_ids(
    text,
    tokenizer,
):
    token_ids = tokenizer.encode(
        text,
        allowed_special={"<|endoftext|>"},
    )

    token_tensor = torch.tensor(
        token_ids,
        dtype=torch.long,
    )

    return token_tensor.unsqueeze(0)


def token_ids_to_text(
    token_ids,
    tokenizer,
):
    token_ids = token_ids.squeeze(0)
    return tokenizer.decode(
        token_ids.tolist()
    )

In [14]:
# define the model evaluation function

def evaluate_model(
    model,
    training_loader,
    validation_loader,
    device,
    evaluation_batches,
):
    model.eval()

    with torch.no_grad():
        training_loss = (
            calculate_loader_loss(
                training_loader,
                model,
                device,
                evaluation_batches,
            )
        )

        validation_loss = (
            calculate_loader_loss(
                validation_loader,
                model,
                device,
                evaluation_batches,
            )
        )

    model.train()

    return training_loss, validation_loss

In [15]:
# define the training-sample generation function
# demonstrates the model's ability to generate text based on a starting prompt

def generate_training_sample(
    model,
    tokenizer,
    device,
    starting_text,
):
    model.eval()

    context_size = (
        model.position_embedding.weight.shape[0]
    )

    input_ids = text_to_token_ids(
        starting_text,
        tokenizer,
    ).to(device)

    with torch.no_grad():
        generated_ids = generate_text_simple(
            model=model,
            token_ids=input_ids,
            max_new_tokens=30,
            context_size=context_size,
        )

    generated_text = token_ids_to_text(
        generated_ids,
        tokenizer,
    )

    print(
        generated_text.replace("\n", " ")
    )

    model.train()

In [16]:
# define the training function

def train_model(
    model,
    training_loader,
    validation_loader,
    optimizer,
    device,
    number_of_epochs,
    evaluation_frequency,
    evaluation_batches,
    starting_text,
    tokenizer,
):

    # initialize lists to track training and validation losses
    # as well as the number of tokens seen
    training_losses = []
    validation_losses = []
    tracked_tokens = []

    tokens_seen = 0
    global_step = -1

    for epoch in range(number_of_epochs):
        model.train()

        for (
            input_batch,
            target_batch,
        ) in training_loader:
            optimizer.zero_grad()

            loss = calculate_batch_loss(
                input_batch,
                target_batch,
                model,
                device,
            )

            loss.backward()
            optimizer.step()

            tokens_seen += input_batch.numel()
            global_step += 1

            if (
                global_step
                % evaluation_frequency
                == 0
            ):
                (
                    training_loss,
                    validation_loss,
                ) = evaluate_model(
                    model,
                    training_loader,
                    validation_loader,
                    device,
                    evaluation_batches,
                )

                training_losses.append(
                    training_loss
                )

                validation_losses.append(
                    validation_loss
                )

                tracked_tokens.append(
                    tokens_seen
                )

                print(
                    f"epoch {epoch + 1} | "
                    f"step {global_step:06d} | "
                    f"training loss "
                    f"{training_loss:.3f} | "
                    f"validation loss "
                    f"{validation_loss:.3f}"
                )

        generate_training_sample(
            model,
            tokenizer,
            device,
            starting_text,
        )

    return (
        training_losses,
        validation_losses,
        tracked_tokens,
    )

In [17]:
# define the optimizer for training the model

learning_rate = 0.0005
weight_decay = 0.1

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)

In [18]:
# set the random seed for reproducibility

torch.manual_seed(123)

number_of_epochs = 10

# train the model and track the training and validation losses
(
    training_losses,
    validation_losses,
    tracked_tokens,
) = train_model(
    model=model,
    training_loader=training_loader,
    validation_loader=validation_loader,
    optimizer=optimizer,
    device=device,
    number_of_epochs=number_of_epochs,
    evaluation_frequency=5,
    evaluation_batches=5,
    starting_text="Every effort moves you",
    tokenizer=tokenizer,
)

epoch 1 | step 000000 | training loss 9.945 | validation loss 10.061
epoch 1 | step 000005 | training loss 7.748 | validation loss 7.954
Every effort moves you,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
epoch 2 | step 000010 | training loss 6.530 | validation loss 6.812
epoch 2 | step 000015 | training loss 5.853 | validation loss 6.565
Every effort moves you, the, and, and, and, the, the, and, and, and, the, and, and, the, and, the
epoch 3 | step 000020 | training loss 5.769 | validation loss 6.510
epoch 3 | step 000025 | training loss 5.485 | validation loss 6.482
Every effort moves you and the                            
epoch 4 | step 000030 | training loss 5.151 | validation loss 6.357
epoch 4 | step 000035 | training loss 5.170 | validation loss 6.438
Every effort moves you in the of the of the of the of the of the of the of the of the picture to          
epoch 5 | step 000040 | training loss 4.436 | validation loss 6.304
Every effort moves you know to                            
epoch 6 | s

In [19]:
# generate text after running

model.eval()

starting_ids = text_to_token_ids(
    "Every effort moves you",
    tokenizer,
).to(device)

with torch.no_grad():
    generated_ids = generate_text_simple(
        model=model,
        token_ids=starting_ids,
        max_new_tokens=50,
        context_size=gpt_config[
            "context_length"
        ],
    )

generated_text = token_ids_to_text(
    generated_ids,
    tokenizer,
)

print(generated_text)

Every effort moves you?"

"Yes--quite insensible to the irony. She wanted him vindicated--and by me!"












"Oh, I saw that, and down the room, in his


In [20]:
# save weights for later models

model_directory = Path("./models")
model_directory.mkdir(
    parents=True,
    exist_ok=True,
)

weights_path = (
    model_directory
    / "gpt_verdict_weights.pth"
)

torch.save(
    model.state_dict(),
    weights_path,
)

print("saved weights to:", weights_path)

saved weights to: models/gpt_verdict_weights.pth


In [21]:
# load the saved weights into a new model instance

loaded_model = gpt_model(gpt_config)

saved_weights = torch.load(
    weights_path,
    map_location="cpu",
    weights_only=True,
)

loaded_model.load_state_dict(
    saved_weights
)

loaded_model = loaded_model.to(device)
loaded_model.eval()

gpt_model(
  (token_embedding): Embedding(50257, 768)
  (position_embedding): Embedding(256, 768)
  (embedding_dropout): Dropout(p=0.1, inplace=False)
  (transformer_blocks): Sequential(
    (0): transformer_block(
      (attention): multi_head_attention(
        (w_query): Linear(in_features=768, out_features=768, bias=False)
        (w_key): Linear(in_features=768, out_features=768, bias=False)
        (w_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (feed_forward): feed_forward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): gelu()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm_1): layer_norm()
      (norm_2): layer_norm()
      (shortcut_dropout): Dropout(p=0.1, inplace=False)
    )
    (1): transformer_block(
     

In [22]:
# weights alone are not enough to resume training, so we save a checkpoint that 
# includes the optimizer state, the model configuration, and the number of epochs completed

checkpoint_path = (
    model_directory
    / "gpt_verdict_checkpoint.pth"
)

torch.save(
    {
        "model_state_dict": (
            model.state_dict()
        ),
        "optimizer_state_dict": (
            optimizer.state_dict()
        ),
        "gpt_config": gpt_config,
        "completed_epochs": (
            number_of_epochs
        ),
        "tokens_seen": (
            tracked_tokens[-1]
        ),
    },
    checkpoint_path,
)

print(
    "saved checkpoint to:",
    checkpoint_path,
)

saved checkpoint to: models/gpt_verdict_checkpoint.pth


In [23]:
# load the checkpoint and resume training from where it left off

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True,
)

resumed_model = gpt_model(
    checkpoint["gpt_config"]
)

resumed_model.load_state_dict(
    checkpoint["model_state_dict"]
)

resumed_model = resumed_model.to(device)

resumed_optimizer = torch.optim.AdamW(
    resumed_model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)

resumed_optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

resumed_model.train()

print(
    "resuming after epoch:",
    checkpoint["completed_epochs"],
)

resuming after epoch: 10
